# Day 8: Retrieval Quality, Walked Through

Week 2 starts here: measuring whether the RAG pipeline from Week 1 actually works
well, not just whether it runs. This notebook builds an evaluation framework,
explains each metric, creates ground truth, visualizes and analyzes results, and
compares retrieval approaches head to head.

Run each cell in order. Plain Python throughout, self-contained.

## Cell 1: Why measuring retrieval matters

You wouldn't ship a car without testing the brakes. A RAG system that only gets
eyeballed on 2-3 example queries can quietly fail on real questions, and nobody
would know until a user hit the failure. Measuring retrieval against known-correct
answers turns "seems fine" into a specific, trackable number.

## Cell 2: Build evaluation framework

The foundation: a dataset of queries with known-relevant documents, and functions
to score a retriever against it.

In [ ]:
import math

class EvaluationDataset:
    def __init__(self):
        self.entries = []
    def add_query(self, query, relevant_ids, relevance_scores=None, difficulty="medium", query_type="factual"):
        if relevance_scores is None:
            relevance_scores = {doc_id: 1 for doc_id in relevant_ids}
        self.entries.append({"query": query, "relevant_ids": set(relevant_ids), "relevance_scores": relevance_scores,
                              "difficulty": difficulty, "query_type": query_type})
    def __len__(self):
        return len(self.entries)
    def __iter__(self):
        return iter(self.entries)

def precision_at_k(ranked_ids, relevant_ids, k):
    top_k = ranked_ids[:k]
    return sum(1 for d in top_k if d in relevant_ids) / len(top_k) if top_k else 0.0

def recall_at_k(ranked_ids, relevant_ids, k):
    if not relevant_ids:
        return 1.0
    top_k = ranked_ids[:k]
    return sum(1 for d in top_k if d in relevant_ids) / len(relevant_ids)

def dcg_at_k(ranked_ids, relevance_scores, k):
    return sum(relevance_scores.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked_ids[:k]))

def ndcg_at_k(ranked_ids, relevance_scores, k):
    actual = dcg_at_k(ranked_ids, relevance_scores, k)
    ideal_order = sorted(relevance_scores, key=lambda d: relevance_scores[d], reverse=True)
    ideal = dcg_at_k(ideal_order, relevance_scores, k)
    return actual / ideal if ideal else 1.0

def reciprocal_rank(ranked_ids, relevant_ids):
    for i, d in enumerate(ranked_ids):
        if d in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0

def average_precision(ranked_ids, relevant_ids):
    if not relevant_ids:
        return 1.0
    hits, precisions = 0, []
    for i, d in enumerate(ranked_ids):
        if d in relevant_ids:
            hits += 1
            precisions.append(hits / (i + 1))
    return sum(precisions) / len(relevant_ids) if precisions else 0.0

def unique_doc_ids_in_order(results):
    # Retrieval returns CHUNKS -- the same document can appear more than
    # once. Metrics assume each relevant id is only "found" once, so we
    # collapse to first-seen unique ids (a real bug from Day 6 otherwise).
    seen, ordered = set(), []
    for r in results:
        doc_id = r["metadata"]["doc_id"]
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered

def evaluate_retriever(retrieve_fn, dataset, k=3):
    per_query = []
    for entry in dataset:
        raw = retrieve_fn(entry["query"], k)
        ranked_ids = unique_doc_ids_in_order(raw)
        per_query.append({
            "query": entry["query"], "difficulty": entry["difficulty"], "query_type": entry["query_type"],
            "precision": precision_at_k(ranked_ids, entry["relevant_ids"], k),
            "recall": recall_at_k(ranked_ids, entry["relevant_ids"], k),
            "ndcg": ndcg_at_k(ranked_ids, entry["relevance_scores"], k),
            "rr": reciprocal_rank(ranked_ids, entry["relevant_ids"]),
            "ap": average_precision(ranked_ids, entry["relevant_ids"]),
        })
    n = len(per_query)
    return {
        "avg_precision": sum(r["precision"] for r in per_query) / n,
        "avg_recall": sum(r["recall"] for r in per_query) / n,
        "avg_ndcg": sum(r["ndcg"] for r in per_query) / n,
        "mrr": sum(r["rr"] for r in per_query) / n,
        "map": sum(r["ap"] for r in per_query) / n,
        "per_query": per_query,
    }

print("Evaluation framework ready.")

## Cell 3: Explain each metric

A worked example showing precision, recall, MAP, NDCG, and MRR each measuring a
genuinely different thing.

In [ ]:
relevant_ids = {2, 5}
relevance_scores = {2: 2, 5: 1}

ranking_a = [2, 5, 1]  # best doc first
ranking_b = [5, 2, 1]  # 2nd-best doc first

print(f"Ranking A: {ranking_a}")
print(f"  Precision@3: {precision_at_k(ranking_a, relevant_ids, 3):.2f}")
print(f"  Recall@3:    {recall_at_k(ranking_a, relevant_ids, 3):.2f}")
print(f"  NDCG@3:      {ndcg_at_k(ranking_a, relevance_scores, 3):.2f}  <- rewards putting the MOST relevant doc first")
print(f"  MRR:         {reciprocal_rank(ranking_a, relevant_ids):.2f}")
print(f"  MAP:         {average_precision(ranking_a, relevant_ids):.2f}")

print(f"\nRanking B: {ranking_b}")
print(f"  Precision@3: {precision_at_k(ranking_b, relevant_ids, 3):.2f}  <- SAME as A")
print(f"  Recall@3:    {recall_at_k(ranking_b, relevant_ids, 3):.2f}  <- SAME as A")
print(f"  NDCG@3:      {ndcg_at_k(ranking_b, relevance_scores, 3):.2f}  <- LOWER: wrong doc ranked first")
print(f"  MRR:         {reciprocal_rank(ranking_b, relevant_ids):.2f}")
print(f"  MAP:         {average_precision(ranking_b, relevant_ids):.2f}")

Precision and recall can't tell A and B apart -- they found the same documents.
NDCG can, because it cares about WHICH document is more relevant being ranked first.

## Cell 4: Create test dataset

Real ground truth against Day 5's actual 6-document set, with difficulty levels.

In [ ]:
DOCUMENTS = [
    {"id": 1, "title": "What is Python", "text": "Python is a high-level programming language known for its simple, readable syntax.\n\nPython was created by Guido van Rossum and first released in 1991."},
    {"id": 2, "title": "What is RAG", "text": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents before generating an answer.\n\nThis approach retrieves relevant information first, then generates a response."},
    {"id": 3, "title": "What is a Vector Database", "text": "A vector database stores data as numerical vectors called embeddings, allowing fast similarity search.\n\nReal vector databases use indexing to keep searches fast at scale."},
    {"id": 5, "title": "Why RAG Matters", "text": "RAG matters because language models can be outdated. Retrieval lets them use fresh information without retraining."},
    {"id": 6, "title": "Keyword Search Basics", "text": "Keyword search finds documents with exact matching words, but misses documents using different words for the same idea."},
]

dataset = EvaluationDataset()
dataset.add_query("What is RAG?", relevant_ids={2}, relevance_scores={2: 2, 5: 1}, difficulty="easy", query_type="factual")
dataset.add_query("What is Python?", relevant_ids={1}, relevance_scores={1: 2}, difficulty="easy", query_type="factual")
dataset.add_query("How do I store embeddings for fast lookup?", relevant_ids={3}, relevance_scores={3: 2}, difficulty="medium", query_type="conceptual")
dataset.add_query("Why would a language model give outdated answers?", relevant_ids={5}, relevance_scores={5: 2, 2: 1}, difficulty="medium", query_type="conceptual")

print(f"Built a dataset with {len(dataset)} labeled queries.")

## Cell 5: Calculate metrics

A simple concept-bucket "embedding" retriever (same idea as Days 2-5), evaluated
against the dataset above.

In [ ]:
CONCEPTS = {
    "programming": ["python", "programming", "language", "syntax"],
    "retrieval":   ["rag", "retrieval", "retrieves", "generation", "lookup", "store"],
    "freshness":   ["outdated", "fresh", "matters", "give", "answers"],
    "search_tech": ["vector", "database", "embeddings", "similarity", "search", "fast"],
}

def text_to_embedding(text):
    words = set(w.strip("?.,!").lower() for w in text.split())
    return [float(sum(1 for w in words if w in cw)) for cw in CONCEPTS.values()]

def cosine_similarity(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    ma, mb = math.sqrt(sum(x*x for x in a)), math.sqrt(sum(y*y for y in b))
    return dot/(ma*mb) if ma and mb else 0.0

def chunk_by_paragraphs(text):
    return [p.strip() for p in text.split("\n\n") if p.strip()]

index = []
for doc in DOCUMENTS:
    for i, chunk in enumerate(chunk_by_paragraphs(doc["text"])):
        emb = text_to_embedding(doc["title"] + " " + chunk)
        index.append((emb, {"doc_id": doc["id"], "title": doc["title"], "chunk_index": i}))

def retrieve(query, k=3):
    q_emb = text_to_embedding(query)
    scored = [(meta, cosine_similarity(q_emb, emb)) for emb, meta in index]
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return [{"metadata": m, "similarity": s} for m, s in scored[:k] if s > 0]

results = evaluate_retriever(lambda q, k: retrieve(q, k), dataset, k=3)
print(f"Precision@3: {results['avg_precision']:.3f}")
print(f"Recall@3:    {results['avg_recall']:.3f}")
print(f"NDCG@3:      {results['avg_ndcg']:.3f}")
print(f"MRR:         {results['mrr']:.3f}")
print(f"MAP:         {results['map']:.3f}")

## Cell 6: Visualize results

A quick text-based bar chart makes the weak queries jump out.

In [ ]:
def bar(value, width=25):
    filled = max(0, min(width, int(value * width)))
    return "#" * filled + "-" * (width - filled)

print("Per-query NDCG:\n")
for r in results["per_query"]:
    print(f"[{r['difficulty']:6s}] {r['query'][:40]:40s} {bar(r['ndcg'])} {r['ndcg']:.2f}")

## Cell 7: Analyze failures

Break down by difficulty to find WHERE the weak spot is.

In [ ]:
by_difficulty = {}
for r in results["per_query"]:
    by_difficulty.setdefault(r["difficulty"], []).append(r["ndcg"])

print("Average NDCG by difficulty:")
for difficulty, scores in sorted(by_difficulty.items(), key=lambda pair: sum(pair[1])/len(pair[1])):
    avg = sum(scores) / len(scores)
    print(f"  {difficulty:8s}: {avg:.3f}  (from {len(scores)} quer{'y' if len(scores)==1 else 'ies'})")

worst = min(results["per_query"], key=lambda r: r["ndcg"])
print(f"\nWorst query: \"{worst['query']}\" (NDCG={worst['ndcg']:.2f}) -- worth debugging with per_query_diagnosis.py's approach.")

## Cell 8: Compare retrievers

A simple keyword-only approach vs. the concept-embedding approach above, on the
same dataset.

In [ ]:
STOPWORDS = {"what", "is", "a", "an", "the", "how", "do", "i", "for", "would"}

def keyword_retrieve(query, k=3):
    query_words = [w.strip("?.,!") for w in query.lower().split() if w not in STOPWORDS]
    scored = []
    for doc in DOCUMENTS:
        for i, chunk in enumerate(chunk_by_paragraphs(doc["text"])):
            chunk_words = set(w.strip("?.,!").lower() for w in (doc["title"] + " " + chunk).split())
            count = sum(1 for w in query_words if w in chunk_words)
            if count > 0:
                scored.append(({"doc_id": doc["id"], "title": doc["title"], "chunk_index": i}, count))
    scored.sort(key=lambda pair: pair[1], reverse=True)
    return [{"metadata": m, "similarity": s} for m, s in scored[:k]]

keyword_results = evaluate_retriever(lambda q, k: keyword_retrieve(q, k), dataset, k=3)
semantic_results = results

print(f"{'Approach':<14} | {'Precision':>9} | {'Recall':>7} | {'NDCG':>6}")
print("-" * 46)
print(f"{'Keyword-only':<14} | {keyword_results['avg_precision']:>9.3f} | {keyword_results['avg_recall']:>7.3f} | {keyword_results['avg_ndcg']:>6.3f}")
print(f"{'Semantic':<14} | {semantic_results['avg_precision']:>9.3f} | {semantic_results['avg_recall']:>7.3f} | {semantic_results['avg_ndcg']:>6.3f}")

## Cell 9: Track progress over time

Simulating a before/after comparison -- the actual shape you'd use to track real
changes over time.

In [ ]:
def compare_runs(current, baseline, current_label, baseline_label):
    print(f"Comparing '{current_label}' vs baseline '{baseline_label}':")
    for metric in ["avg_precision", "avg_recall", "avg_ndcg", "mrr", "map"]:
        delta = current[metric] - baseline[metric]
        direction = "IMPROVED" if delta > 1e-9 else ("REGRESSED" if delta < -1e-9 else "UNCHANGED")
        print(f"  {metric:14s}: {baseline[metric]:.3f} -> {current[metric]:.3f}  ({direction}, {delta:+.3f})")

compare_runs(semantic_results, keyword_results, "semantic (concept embeddings)", "keyword-only")

## Cell 10: Key takeaways

- You can't improve what you don't measure -- evaluation turns "seems to work" into
  a specific, trackable number.
- Different metrics measure different things: precision (avoid noise), recall
  (don't miss anything), NDCG (rank the BEST result first, not just find it), MRR
  (how fast is the first good answer), MAP (quality across multiple hits).
- A single averaged score hides where problems actually are -- break results down
  by query difficulty and type to find real patterns.
- Good ground truth (real queries, honestly labeled, across difficulty levels) is
  the single most important input -- a perfect metric implementation is worthless
  against bad ground truth.
- Always collapse retrieval results to unique document IDs before scoring --
  duplicate chunks from the same document can silently inflate metrics.
- Evaluate every pipeline stage (retrieval alone, then retrieval+ranking)
  separately, so you know exactly which stage a change actually affected.
- Don't assume one retrieval approach is always better -- measure it on YOUR data,
  since edge cases in a specific embedding or matching approach can flip which
  method actually wins.